In [1]:
from pathlib import Path
root = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/train_hires")
niis = list(root.rglob("*.nii.gz")) + list(root.rglob("*.nii"))
print("Found:", len(niis), "NIfTI files")
for p in niis[:5]:
    print("  ", p.name)


Found: 1292 NIfTI files
   sub-M2101_ses-1309_lesion_mask_MNI_clean.nii.gz
   sub-r009s072_ses-1_lesion_mask_MNI_clean.nii.gz
   sub-M2299_ses-424_T1w_MNI_norm.nii.gz
   sub-r038s091_ses-1_lesion_mask_MNI_clean.nii.gz
   sub-r047s039_ses-1_lesion_mask_MNI_clean.nii.gz


In [ ]:
# === Fresh, OOM-proof training launcher (from scratch) ===
from pathlib import Path
import importlib.util, os, sys, gc, traceback, time, shlex, subprocess
import tensorflow as tf
from tensorflow.keras import mixed_precision

# ---------- CONFIG YOU MAY TWEAK ----------
CUDA_ID        = "0"  # single GPU
RUN_ROOT       = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined")
MODULE_PATH    = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
DATA_DIR       = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/train_hires")

INPUT_SHAPE    = (192, 224, 192, 1)
BATCH_SIZE     = 1          # OOM-safe
BASE_FILTERS   = 6
SAM_HEADS      = 2
AUG_INTENSITY  = 0.2        # light aug for generalization (set 0 to disable)
VAL_SPLIT      = 0.10
TOTAL_EPOCHS   = 60
INITIAL_EPOCH  = 0          # fresh schedule
# ------------------------------------------

# --- Derived run paths (new unique run folder) ---
RUN_ID      = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR     = RUN_ROOT / "runs" / RUN_ID
MODEL_DIR   = RUN_DIR / "models"
CALLBACKS_DIR = RUN_DIR / "callbacks"
LOG_DIR     = RUN_DIR / "logs"
for d in (MODEL_DIR, CALLBACKS_DIR, LOG_DIR): d.mkdir(parents=True, exist_ok=True)

# --- Environment (verbose TF + safe GPU alloc) ---
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_ID
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"          # show TF logs
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # avoid sudden OOM spikes
os.environ["SMARTSOTA_LOG_DIR"] = str(LOG_DIR)    # for your training file's logging

# --- Fresh graph/session ---
tf.keras.backend.clear_session(); gc.collect()

# --- Mixed precision ---
mixed_precision.set_global_policy("mixed_float16")

# --- Tee logs to console + file ---
class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, data):
        for s in self.streams: s.write(data); s.flush()
        return len(data)
    def flush(self):
        for s in self.streams: s.flush()

log_file = open(LOG_DIR / "train_stdout_stderr.log", "a", buffering=1)
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

print("Run ID:", RUN_ID)
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print("set_memory_growth failed:", e)

# --- Import your training module (pre-inject tf so early tf.* calls work) ---
spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg = importlib.util.module_from_spec(spec)
seg.tf = tf  # <- important to avoid NameError if module uses tf before importing
spec.loader.exec_module(seg)

# --- CRITICAL: prevent MirroredStrategy on a single GPU (save VRAM) ---
seg.strategy = tf.distribute.get_strategy()
print("Override strategy to:", type(seg.strategy).__name__)

# --- Launch training (fresh init; no checkpoints loaded) ---
try:
    history = seg.train_dynamic_model(
        DATA_DIR=DATA_DIR,
        MODEL_DIR=MODEL_DIR,
        CALLBACKS_DIR=CALLBACKS_DIR,
        TOTAL_EPOCHS=TOTAL_EPOCHS,
        INITIAL_EPOCH=INITIAL_EPOCH,        # start schedule from 0
        LOAD_WEIGHTS_FROM=None,             # <-- ensure fresh start
        RESUME_FROM_LATEST=False,           # <-- don't resume
        VALIDATION_SPLIT=VAL_SPLIT,
        BATCH_SIZE=BATCH_SIZE,              # OOM-safe
        BASE_FILTERS=BASE_FILTERS,
        SAM_HEADS=SAM_HEADS,
        RESAMPLE_TO_TARGET=True,
        AUGMENTATION_INTENSITY=AUG_INTENSITY,
        INPUT_SHAPE=INPUT_SHAPE
    )
    print("Done. Logged metrics:", list(getattr(history, "history", {}).keys()))

except Exception as e:
    print("\n================= UNCAUGHT EXCEPTION =================")
    traceback.print_exc()
    print("======================================================\n")
    try:
        print("Last few GPU snapshots:")
        for _ in range(3):
            subprocess.run(shlex.split("nvidia-smi"), check=False)
            time.sleep(1)
    except Exception:
        pass
    raise
finally:
    try: log_file.flush()
    except Exception: pass


2025-11-05 16:50:09.924489: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Run ID: 20251105_165011
TF: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Strategy: _DefaultDistributionStrategy
Override strategy to: _DefaultDistributionStrategy


2025-11-05 16:50:11,807 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-05 16:50:11,808 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-05 16:50:11,808 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 1
2025-11-05 16:50:11,810 - SmartSOTA_Dynamic - INFO - 🧭 INPUT_SHAPE set to: (192, 224, 192, 1)
2025-11-05 16:50:11,810 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
I0000 00:00:1762386611.909673  449494 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1762386611.910702  449494 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
2025-11-05 16:50:11,913 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=0.73GB | GPU mem track

2025-11-05 16:50:54,863 - SmartSOTA_Dynamic - INFO - Model: "SmartSOTA_Dynamic"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 192, 224,  │          0 │ -                 │
│ (InputLayer)        │ 192, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_block │ (None, 192, 224,  │      1,194 │ input_layer[0][0] │
│ (ResidualConvBlock) │ 192, 6)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vision_mamba_block  │ (None, 192, 224,  │      4,050 │ residual_conv_bl… │
│ (VisionMambaBlock)  │ 192, 6)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

Epoch 1/60


...

2025-11-05 17:58:44,108 - SmartSOTA_Dynamic - INFO - Memory at epoch_59_end: CPU=11.49GB | GPU mem tracking failed | Disk: 1244.8GB free


Epoch 60: dice=0.2366 val_dice=0.5284 loss=0.3137 val_loss=0.1960 lr=6.25e-06
258/258 ━━━━━━━━━━━━━━━━━━━━ 67s 259ms/step - dice_coefficient: 0.2366 - loss: 0.3137 - val_dice_coefficient: 0.5284 - val_loss: 0.1960 - learning_rate: 6.2500e-06


2025-11-05 17:58:44,694 - SmartSOTA_Dynamic - INFO - 💾 Saved final weights to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/models/smart_sota_dynamic_20251105_165011.final.weights.h5


2025-11-05 17:58:45,095 - SmartSOTA_Dynamic - INFO - 💾 Saved full model to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/models/smart_sota_dynamic_20251105_165011.keras
2025-11-05 17:58:45,095 - SmartSOTA_Dynamic - INFO - 🏁 Training complete.


Done. Logged metrics: ['dice_coefficient', 'loss', 'val_dice_coefficient', 'val_loss', 'learning_rate']
Last epoch: 60 val_dice = 0.5283551216125488
Best val_dice: 0.5426192879676819 at epoch 58
Last epoch: 60 val_dice = 0.5283551216125488
Best val_dice: 0.5426192879676819 at epoch 58


In [5]:
import os, json, csv, glob, time
from pathlib import Path

# --- set your base path (same as before) ---
RUN_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined")

# 1) Find latest run dir: .../runs/<timestamp>/
runs_root = RUN_ROOT / "runs"
run_dirs = sorted([p for p in runs_root.glob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime)
if not run_dirs:
    raise RuntimeError(f"No runs found under {runs_root}")
RUN_DIR = run_dirs[-1]
print("Using RUN_DIR:", RUN_DIR)

callbacks_dir = RUN_DIR / "callbacks"
logs_dir = RUN_DIR / "logs"

# 2) Try CSV first
csv_path = callbacks_dir / "history.csv"
last_val = best_val = None
best_epoch = None

if csv_path.exists():
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        if rows:
            last = rows[-1]
            # keys are like 'val_dice_coefficient'; be robust to missing
            last_val = float(last.get("val_dice_coefficient")) if last.get("val_dice_coefficient") not in (None, "", "nan") else None
            # compute best
            best_val = -1.0
            for i, r in enumerate(rows, start=1):
                v = r.get("val_dice_coefficient")
                if v not in (None, "", "nan"):
                    v = float(v)
                    if v > best_val:
                        best_val = v
                        best_epoch = i
            print(f"[CSV] epochs logged: {len(rows)}")
            print(f"[CSV] last val_dice: {last_val}")
            print(f"[CSV] best val_dice: {best_val} at epoch {best_epoch}")
else:
    print(f"[CSV] not found at {csv_path}")

# 3) Fallback: JSON snapshot (if present)
json_path = callbacks_dir / "artifacts" / "history_epoch.json"
if (last_val is None or best_val is None) and json_path.exists():
    with open(json_path) as f:
        h = json.load(f)
    vals = h.get("val_dice_coefficient", [])
    if vals:
        last_val = vals[-1]
        best_val = max(vals)
        best_epoch = int(vals.index(best_val)) + 1
        print(f"[JSON] last val_dice: {last_val}")
        print(f"[JSON] best val_dice: {best_val} at epoch {best_epoch}")
    else:
        print("[JSON] val_dice_coefficient missing/empty")
elif (last_val is None or best_val is None):
    print(f"[JSON] not found at {json_path}")

# 4) Point you to the log for full text
log_path = logs_dir / "train_stdout_stderr.log"
print("Log file:", log_path, "(exists:", log_path.exists(), ")")

# 5) Show where the best checkpoint lives
best_ckpt = callbacks_dir / "best_model_dynamic.weights.h5"
print("Best checkpoint:", best_ckpt, "(exists:", best_ckpt.exists(), ")")


Using RUN_DIR: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011
[CSV] epochs logged: 60
[CSV] last val_dice: 0.5283551216125488
[CSV] best val_dice: 0.5426192879676819 at epoch 58
Log file: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/logs/train_stdout_stderr.log (exists: True )
Best checkpoint: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/callbacks/best_model_dynamic.weights.h5 (exists: True )


In [13]:
# --- Load your training module so custom objects are available ---
import importlib.util, pathlib, json, time
from pathlib import Path

# Path to the training file that defines the custom layers/losses
MODULE_PATH = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)  # seg now has ResidualConvBlock, VisionMambaBlock, etc.

# --- Path to the full saved Keras model (.keras) you want to inspect ---
MODEL_PATH = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/models/smart_sota_dynamic_20251105_165011.keras")

# --- Load the model with custom objects ---
from keras.saving import load_model  # if this errors, use: from tensorflow.keras.models import load_model
m = load_model(
    MODEL_PATH,
    compile=False,
    custom_objects={
        "ResidualConvBlock": seg.ResidualConvBlock,
        "VisionMambaBlock": seg.VisionMambaBlock,
        "SAM2Attention":    seg.SAM2Attention,
        "CombinedLoss":     seg.CombinedLoss,
        "dice_coefficient": seg.dice_coefficient,
        "dice_loss":        seg.dice_loss,
        "boundary_loss":    seg.boundary_loss,
    }
)

# --- Dump a small “effective config” JSON next to the model ---
cfg_effective = {
    "INPUT_SHAPE": tuple(m.input_shape[1:]),
    "OUTPUT_SHAPE": tuple(m.output_shape[1:]),
    "PARAMS": int(m.count_params()),
    "MODEL_NAME": m.name,
}
out = MODEL_PATH.with_name(f"config_effective_{time.strftime('%Y%m%d_%H%M%S')}.json")
with open(out, "w") as f:
    json.dump(cfg_effective, f, indent=2)

print("Loaded model:", MODEL_PATH.name)
print("Effective config written to:", out)
print(cfg_effective)


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


2025-11-06 10:05:37,731 - tensorflow - INFO - Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
2025-11-06 10:05:37,735 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-06 10:05:37,736 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-06 10:05:37,737 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 2


Strategy: MirroredStrategy
Loaded model: smart_sota_dynamic_20251105_165011.keras
Effective config written to: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/models/config_effective_20251106_100538.json
{'INPUT_SHAPE': (192, 224, 192, 1), 'OUTPUT_SHAPE': (192, 224, 192, 1), 'PARAMS': 1568455, 'MODEL_NAME': 'SmartSOTA_Dynamic'}


In [9]:
from pathlib import Path
import glob, tensorflow as tf

MODELS_DIR = RUN_DIR / "models"
candidates = sorted(glob.glob(str(MODELS_DIR / "smart_sota_dynamic_*.keras")))
print("Found full models:", candidates[-3:])

if candidates:
    latest_full = candidates[-1]
    print("Loading full model:", latest_full)
    # supply custom objects so Keras can deserialize your layers/losses
    from tensorflow.keras.models import load_model
    full = load_model(
        latest_full,
        compile=False,
        custom_objects={
            "ResidualConvBlock": seg.ResidualConvBlock,
            "VisionMambaBlock": seg.VisionMambaBlock,
            "SAM2Attention": seg.SAM2Attention,
            "CombinedLoss": seg.CombinedLoss,
            "dice_coefficient": seg.dice_coefficient,
            "dice_loss": seg.dice_loss,
            "boundary_loss": seg.boundary_loss,
        },
    )
    full.summary()
    print("Loaded full model ✅")
else:
    print("No full .keras model found in", MODELS_DIR)


Found full models: ['/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/models/smart_sota_dynamic_20251105_165011.keras']
Loading full model: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/models/smart_sota_dynamic_20251105_165011.keras


Model: "SmartSOTA_Dynamic"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 192, 224,  │          0 │ -                 │
│ (InputLayer)        │ 192, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_block │ (None, 192, 224,  │      1,194 │ input_layer[0][0] │
│ (ResidualConvBlock) │ 192, 6)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vision_mamba_block  │ (None, 192, 224,  │      4,050 │ residual_conv_bl… │
│ (VisionMambaBlock)  │ 192, 6)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d       │ (None, 96, 112,   │          0 │ vision_mamba_blo… │
│ (MaxPooling3D)      │ 96, 6)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_bloc… │ (None, 96, 112,   │      6,012 │ max_pooling3d[0]… │
│ (ResidualConvBlock) │ 96, 12)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vision_mamba_block… │ (None, 96, 112,   │     16,164 │ residual_conv_bl… │
│ (VisionMambaBlock)  │ 96, 12)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_1     │ (None, 48, 56,    │          0 │ vision_mamba_blo… │
│ (MaxPooling3D)      │ 48, 12)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_bloc… │ (None, 48, 56,    │     23,832 │ max_pooling3d_1[… │
│ (ResidualConvBlock) │ 48, 24)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vision_mamba_block… │ (None, 48, 56,    │     64,584 │ residual_conv_bl… │
│ (VisionMambaBlock)  │ 48, 24)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_2     │ (None, 24, 28,    │          0 │ vision_mamba_blo… │
│ (MaxPooling3D)      │ 24, 24)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_bloc… │ (None, 24, 28,    │     94,896 │ max_pooling3d_2[… │
│ (ResidualConvBlock) │ 24, 48)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vision_mamba_block… │ (None, 24, 28,    │    258,192 │ residual_conv_bl… │
│ (VisionMambaBlock)  │ 24, 48)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_3     │ (None, 12, 14,    │          0 │ vision_mamba_blo… │
│ (MaxPooling3D)      │ 12, 48)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_bloc… │ (None, 12, 14,    │    378,720 │ max_pooling3d_3[… │
│ (ResidualConvBlock) │ 12, 96)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sam2_attention      │ (None, 12, 14,    │     37,344 │ residual_conv_bl… │
│ (SAM2Attention)     │ 12, 96)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling3d       │ (None, 24, 28,    │          0 │ sam2_attention[0… │
│ (UpSampling3D)      │ 24, 96)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 24, 28,    │          0 │ up_sampling3d[0]

 Total params: 1,568,455 (5.98 MB)

 Trainable params: 1,568,455 (5.98 MB)

 Non-trainable params: 0 (0.00 B)

Loaded full model ✅


In [11]:
import numpy as np, tensorflow as tf, json
from pathlib import Path

# Assume you still have:
#   - full : the loaded full Keras model from the .keras file
#   - seg  : your training module (imported)
#   - RUN_DIR from earlier cell
# If not, re-import your module and reload the full model like you just did.

# 1) Rebuild a config that matches the loaded model’s input shape
inp = full.inputs[0].shape  # (None, D, H, W, C)
INPUT_SHAPE = (int(inp[1]), int(inp[2]), int(inp[3]), int(inp[4]))

cfg = seg.DynamicTrainingConfig(
    DATA_DIR=Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/train_hires"),
    INPUT_SHAPE=INPUT_SHAPE,
    VALIDATION_SPLIT=0.10,
    BATCH_SIZE=1,                 # evaluate 1 volume at a time (clear per-case)
    MODEL_DIR=RUN_DIR / "models",
    CALLBACKS_DIR=RUN_DIR / "callbacks",
)

# 2) Get the SAME dataset & split function you trained with
pairs, lesion_presence = seg.load_generic_dataset(cfg)
train_pairs, val_pairs = seg.create_stratified_splits(
    pairs, lesion_presence, batch_size=cfg.BATCH_SIZE, test_size=cfg.VALIDATION_SPLIT
)

# Helper: compute Dice (soft / hard)
def soft_dice(y_true, y_pred, eps=1e-6):
    y_true = y_true.astype(np.float32)
    y_pred = np.clip(y_pred.astype(np.float32), 1e-7, 1.0 - 1e-7)
    inter  = np.sum(y_true * y_pred)
    denom  = np.sum(y_true) + np.sum(y_pred)
    return (2*inter + eps) / (denom + eps)

def hard_dice(y_true, y_prob, thr=0.5, eps=1e-6):
    y_bin = (y_prob >= thr).astype(np.float32)
    inter = np.sum(y_true * y_bin)
    denom = np.sum(y_true) + np.sum(y_bin)
    return (2*inter + eps) / (denom + eps)

# 3) Iterate the validation set, compute per-case scores
per_case_soft = []
per_case_hard = []
thr = 0.5

for img_path, msk_path in val_pairs:
    # load and preprocess exactly like your generator
    img = seg._load_and_preprocess_image(str(img_path), cfg.INPUT_SHAPE[:-1])   # (D,H,W)
    msk = seg._load_and_preprocess_mask(str(msk_path), cfg.INPUT_SHAPE[:-1])    # (D,H,W)

    x = np.zeros((1, *cfg.INPUT_SHAPE), dtype=np.float32)
    y = np.zeros((1, *cfg.INPUT_SHAPE), dtype=np.float32)
    x[0, ..., 0] = img
    y[0, ..., 0] = msk

    # predict
    prob = full.predict(x, verbose=0)  # (1,D,H,W,1)
    prob = prob[0, ..., 0]
    gt   = y[0, ..., 0]

    per_case_soft.append(soft_dice(gt, prob))
    per_case_hard.append(hard_dice(gt, prob, thr=thr))

# 4) Macro (per-case) averages
macro_soft = float(np.mean(per_case_soft))
macro_hard = float(np.mean(per_case_hard))

# 5) Micro (global) Dice over the whole val set
all_probs = []
all_gts   = []
for img_path, msk_path in val_pairs:
    img = seg._load_and_preprocess_image(str(img_path), cfg.INPUT_SHAPE[:-1])
    msk = seg._load_and_preprocess_mask(str(msk_path), cfg.INPUT_SHAPE[:-1])
    x = np.zeros((1, *cfg.INPUT_SHAPE), dtype=np.float32)
    x[0, ..., 0] = img
    p = full.predict(x, verbose=0)[0, ..., 0]
    all_probs.append(p.astype(np.float32).ravel())
    all_gts.append(msk.astype(np.float32).ravel())

all_probs = np.concatenate(all_probs, axis=0)
all_gts   = np.concatenate(all_gts, axis=0)

micro_soft = soft_dice(all_gts, all_probs)
micro_hard = hard_dice(all_gts, all_probs, thr=thr)

print(f"Per-case (macro) soft Dice: {macro_soft:.4f}")
print(f"Per-case (macro) hard Dice @ {thr}: {macro_hard:.4f}")
print(f"Global (micro) soft Dice  : {micro_soft:.4f}")
print(f"Global (micro) hard Dice @ {thr}: {micro_hard:.4f}")
print(f"Val set size: {len(val_pairs)} cases")


2025-11-06 09:54:58,660 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2025-11-06 09:54:58,662 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=1.02GB | GPU mem tracking failed | Disk: 1244.6GB free
2025-11-06 09:54:58,716 - SmartSOTA_Dynamic - INFO - 📁 Single-folder mode: 646 images, 646 masks in /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/train_hires
2025-11-06 09:54:58,717 - SmartSOTA_Dynamic - INFO - Found 646 image files and 646 mask files
2025-11-06 09:55:42,855 - SmartSOTA_Dynamic - INFO - 📊 Created 323 image–mask pairs
2025-11-06 09:55:42,856 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-11-06 09:55:42,856 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=1.05GB | GPU mem tracking failed | Disk: 1244.6GB free
2025-11-06 09:55:42,857 - SmartSOTA_Dynamic - INFO - 🧮 Dataset split: Train=291 (90.

Per-case (macro) soft Dice: 0.5284
Per-case (macro) hard Dice @ 0.5: 0.5310
Global (micro) soft Dice  : 0.6712
Global (micro) hard Dice @ 0.5: 0.6744
Val set size: 32 cases


# Validation Metrics — What They Mean

**Val set size:** `32` cases  
**Lesion presence:** `100%` (every case has positive voxels)

## Per-case (Macro) Soft Dice: `0.5284`

**How computed (per subject \(i\)):**
\[
\mathrm{Dice}^{\text{soft}}_i \;=\; \frac{2\,\sum (p_i \cdot y_i)}{\sum p_i \;+\; \sum y_i}
\]

**Macro average across subjects:**
\[
\text{Macro Soft Dice} \;=\; \frac{1}{N}\sum_{i=1}^{N}\mathrm{Dice}^{\text{soft}}_i
\]

**Interpretation:** Patient-level average Dice on **probabilities** (no binarization). Each patient counts **equally**, so small/large lesions have equal weight in the final average.

---

## Per-case (Macro) Hard Dice @ 0.5: `0.5310`

**Threshold then compute per subject:**
\[
\hat{y}_i \;=\; \mathbb{1}[\,p_i \ge 0.5\,], 
\qquad
\mathrm{Dice}^{\text{hard}}_i \;=\; \frac{2\,\sum (\hat{y}_i \cdot y_i)}{\sum \hat{y}_i \;+\; \sum y_i}
\]

**Macro average across subjects:**
\[
\text{Macro Hard Dice@0.5} \;=\; \frac{1}{N}\sum_{i=1}^{N}\mathrm{Dice}^{\text{hard}}_i
\]

**Interpretation:** Patient-level average on **binarized** masks at threshold 0.5. Similar to soft Dice here ⇒ probabilities are reasonably calibrated near 0.5.

---

## Global (Micro) Soft Dice: `0.6712`

**Concatenate all voxels across all subjects and compute once:**
\[
\text{Micro Soft Dice} 
\;=\; \frac{2\,\sum_i \sum (p_i \cdot y_i)}{\sum_i \sum p_i \;+\; \sum_i \sum y_i}
\]

**Interpretation:** **Volume-weighted** Dice. Large lesions contribute more. Higher than macro ⇒ the model performs better on larger lesions and/or large lesions dominate the cohort.

---

## Global (Micro) Hard Dice @ 0.5: `0.6744`

**Concatenate after thresholding:**
\[
\hat{y}_i \;=\; \mathbb{1}[\,p_i \ge 0.5\,], 
\qquad
\text{Micro Hard Dice@0.5} 
\;=\; \frac{2\,\sum_i \sum (\hat{y}_i \cdot y_i)}{\sum_i \sum \hat{y}_i \;+\; \sum_i \sum y_i}
\]

**Interpretation:** What your **binarized** outputs achieve when the entire validation set is treated as one big volume.

---

## Why Macro < Micro Here

- **Macro (per-case)** gives **equal weight per patient** → sensitive to small lesions and tough cases.  
- **Micro (global)** gives **more weight to larger lesions** → tends to be higher when big lesions are easier.

Gap (~0.53 vs ~0.67) ⇒ small lesions are relatively harder (common in stroke).

---

## Practical Notes

- **Clinical reporting:** Prefer **macro hard Dice @ chosen threshold** for fairness across patients; include micro for context.
- **Threshold choice:** You evaluated @ `0.5`. Consider a quick sweep (e.g., 0.2–0.6) and select the threshold that maximizes **macro Dice** or your desired recall/precision trade-off.
- **Calibration:** Soft ≈ Hard at 0.5 suggests reasonable calibration; still worth validating with a threshold scan.
- **Improvement ideas (if you want higher macro Dice / small-lesion sensitivity):**
  - Add **focal** term or small-lesion weighting to the loss.
  - Sample/oversample patches with positives or hard examples.
  - Keep **light** augmentation to aid generalization; monitor macro Dice.  

---

**TL;DR**  
- **Macro (per-case)** ≈ **0.53** → average patient Dice.  
- **Micro (global)** ≈ **0.67** → boosted by large lesions.  
- The gap points to small-lesion difficulty; tune threshold and loss/sampling to improve parity.
